# 04b - Modeling: Grid Search Hyperparameter## TujuanMenentukan konfigurasi terbaik untuk jumlah topik K dan prior Dirichlet(alpha, eta) melalui grid search exhaustive.## Asal (Input)`data/intermediate/dataset_preprocessed.csv` + `dictionary.gensim`## Keluaran (Output)`model/hyperparameter_results.csv` - hasil 84 konfigurasi## Ruang Grid| Parameter | Nilai yang diuji ||:---|:---|| K | 4, 5, 6, 7, 8, 9, 10 (7 nilai) || Alpha | symmetric, asymmetric, 0.01, 0.1 (4 nilai) || Eta | symmetric, 0.01, 0.1 (3 nilai) |**Total: 7 x 4 x 3 = 84 konfigurasi**, masing-masing dengan passes=10.## Catatan Penting tentang Pemilihan KGrid search ini menunjukkan **K=9 mencapai C_V tertinggi (0,4314)**, bukan K=8.K=8 dipilih atas pertimbangan gabungan: jumlah dokumen per topik lebih memadai(minimum 21) dan struktur topik lebih mudah diinterpretasikan. Hal inidijelaskan secara transparan pada Bab IV skripsi.

In [ ]:
import sys, io as _iofrom pathlib import Path_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(_ROOT))try:    sys.stdout.reconfigure(encoding='utf-8', errors='replace')except Exception:    passimport pandas as pd, numpy as npfrom src import config as CC.ensure_dirs()print(C.describe())

## Langkah 1 - Jalankan Grid Search

In [ ]:
from src import grid_searchgrid_search.main()

## Langkah 2 - Analisis Hasil: C_V Tertinggi per K

In [ ]:
hp = pd.read_csv(C.HYPERPARAMS_CSV)print(f'Total konfigurasi: {len(hp)}')print()print('=== C_V MAKSIMUM PER K ===')for k, grp in hp.groupby('k'):    print(f'  K={k:2d}  C_V max = {grp["coherence_cv"].max():.4f}   '          f'C_V mean = {grp["coherence_cv"].mean():.4f}')

## Langkah 3 - 10 Konfigurasi Teratas

In [ ]:
top = hp.sort_values('coherence_cv', ascending=False).head(10)print(top[['k','alpha','eta','coherence_cv','coherence_umass']].to_string(index=False))

## Langkah 4 - Elbow Curve

In [ ]:
import matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltsweep = hp.groupby('k')['coherence_cv'].agg(['max','mean']).reset_index()fig, ax = plt.subplots(figsize=(9, 4.5))ax.plot(sweep['k'], sweep['max'], '-o', color='#1A56A0', linewidth=2, label='Max C_V')ax.plot(sweep['k'], sweep['mean'], '--o', color='#5B9BD5', alpha=0.7, label='Mean C_V')ax.axvline(8, color='#00C48C', linestyle='--', linewidth=2)ax.text(8.1, sweep['max'].max()*0.98, 'Model Final K=8', color='#00C48C', fontsize=9)ax.set_xlabel('Jumlah Topik (K)'); ax.set_ylabel('Coherence Score (C_V)')ax.set_title('Grid Search 84 Konfigurasi')ax.set_xticks(sorted(hp['k'].unique())); ax.grid(alpha=0.3); ax.legend()plt.tight_layout(); plt.show()

## Ringkasan Fase 04b - Grid Search| K | C_V maksimum | Catatan ||---:|---:|:---|| 4 | 0,3605 | || 5 | 0,3756 | || 6 | 0,3909 | || 7 | 0,4142 | || 8 | 0,3871 | **Dipilih** - struktur lebih seimbang || 9 | **0,4314** | C_V tertinggi, tapi topik terlalu ringkas || 10 | 0,3993 | |Pemilihan K=8 adalah keputusaninterpretif combined: bukan semata C_Vtertinggi, melainkan menimbang koherensi, jumlah dokumen per topik,dan interpretabilitas.

## Peta Artefak| Artefak | Dibuat oleh | Consumed by ||:---|:---|:---|| `model/hyperparameter_results.csv` | Notebook 04b | Notebook 04 (-informed) |